[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/planessoria-ui/ModelVinya/blob/main/auto_labeling/sam3_to_modelvinya.ipynb)

# 🍇 ModelVinya · Etiquetatge automàtic amb SAM 3

Genera un projecte **`.json` de ModelVinya** (cercles = baies) a partir de fotos, fent servir **SAM 3** (Meta). Després l'obres a l'anotador (https://planessoria-ui.github.io/ModelVinya/) per **revisar, posar escala i exportar**.

### Abans de començar
1. **Entorn amb GPU**: *Entorn d'execució → Canviar el tipus d'entorn → GPU*.
2. **Accés als checkpoints** (gated): demana accés a https://huggingface.co/facebook/sam3 i crea un token a https://huggingface.co/settings/tokens

### Ordre
1. Executa **PAS 1 → PAS 4** (instal·lar, login, model, funcions). Sempre.
2. Tria **una opció**:
   - **OPCIÓ A — Safata amb regle** (per calibrar/entrenar amb referència real en mm).
   - **OPCIÓ B — Racims a la planta** (camp, sense referència).


In [ ]:
# PAS 1) Instal·lar SAM 3 (+ versions compatibles amb numpy 1.26 que SAM 3 necessita)
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip -q install -e .
!pip -q install huggingface_hub "numpy==1.26.4" "opencv-python-headless==4.10.0.84"

### ⚠️ IMPORTANT: reinicia l'entorn després del PAS 1
Colab arrenca amb numpy 2 carregat; SAM 3 necessita numpy 1.26. Després d'executar el **PAS 1**, ves a **Entorno de ejecución → Reiniciar entorno** i després continua amb el **PAS 2** (NO tornis a executar el PAS 1). Si t'ho saltes, el PAS 3 dona l'error `numpy.dtype size changed`.

In [ ]:
# PAS 2) Accés als checkpoints (gated). Enganxa el teu token de Hugging Face.
from huggingface_hub import login
login()

In [ ]:
# PAS 3) Carregar el model SAM 3 (descarrega el checkpoint facebook/sam3 la primera vegada)
import glob
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# El install editable (-e) de vegades no localitza el tokenizer BPE -> el passem explícitament
_bpe = glob.glob('/content/sam3/**/bpe_simple_vocab_16e6.txt.gz', recursive=True)
bpe_path = _bpe[0] if _bpe else None
print('Tokenizer BPE:', bpe_path)

model = build_sam3_image_model(bpe_path=bpe_path)
processor = Sam3Processor(model)
print('SAM 3 carregat \u2714')

In [ ]:
# PAS 4) Funcions auxiliars (comunes a totes les opcions). Executa-la sempre.
import numpy as np, base64, json, mimetypes, os, io, torch
from PIL import Image, ImageOps

def to_np(x):
    """Tensor de PyTorch (GPU/bfloat16) o array -> numpy float32 a CPU."""
    if hasattr(x, "detach"):
        x = x.detach().cpu().float()
    return np.asarray(x)

def to_binary(mask):
    arr = np.squeeze(to_np(mask))
    if arr.dtype == bool:
        return arr
    return arr > 0.5 if arr.max() <= 1.0 else arr > 0

def mask_to_circle(binary):
    ys, xs = np.nonzero(binary)
    if len(xs) == 0:
        return None
    area = float(len(xs))            # Diametre equivalent: D = 2*sqrt(A/pi)
    return {"cx": round(float(xs.mean()), 2), "cy": round(float(ys.mean()), 2),
            "r": round(float(np.sqrt(area / np.pi)), 2), "area_px": area}

def open_img(path):
    # aplica la rotacio EXIF perque les coordenades coincideixin amb el que mostra el navegador
    return ImageOps.exif_transpose(Image.open(path)).convert('RGB')

def data_url(img):
    # re-codifica la imatge (ja girada) -> el .json mostra exactament el que SAM ha vist
    buf = io.BytesIO(); img.convert('RGB').save(buf, format='JPEG', quality=92)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

def detect(pil_img, prompt):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        st  = processor.set_image(pil_img)
        out = processor.set_text_prompt(state=st, prompt=prompt)
    return to_np(out['masks']), to_np(out['scores']).reshape(-1)

def preview(im0):
    import matplotlib.pyplot as plt
    img = open_img(im0['nombre'])
    fig, ax = plt.subplots(figsize=(8, 10)); ax.imshow(img)
    if im0.get('racimo'):
        xs = [p[0] for p in im0['racimo']['puntos']] + [im0['racimo']['puntos'][0][0]]
        ys = [p[1] for p in im0['racimo']['puntos']] + [im0['racimo']['puntos'][0][1]]
        ax.plot(xs, ys, 'r-', lw=2)
    for b in im0['bayas']:
        ax.add_patch(plt.Circle((b['cx'], b['cy']), b['r'], fill=False, color='lime', lw=1.2))
    ax.set_title(f"{len(im0['bayas'])} baies"); ax.axis('off'); plt.show()

def save_project(images_out, name='modelvinya_sam3_proyecto.json'):
    project = {"app": "ModelVinya", "version": 1,
               "savedAt": __import__('datetime').datetime.utcnow().isoformat(), "images": images_out}
    with open(name, 'w') as f:
        json.dump(project, f)
    print(f"Guardat: {name} | {len(images_out)} imatges | {sum(len(i['bayas']) for i in images_out)} baies")

EMPTY_FICHA  = {"id_racimo": "", "fecha": "", "variedad": "", "fase_fenologica": "",
                "tratamiento": "", "vigor": "", "orientacion": "", "sistema_conduccion": ""}
EMPTY_VERDAD = {"N_total_real": "", "Diam_real_mm": "", "Peso_baya_real_g": "", "Peso_racimo_real_g": ""}
print('Funcions auxiliars carregades ✔')

---
## 🟦 OPCIÓ A — Fotos de safata (amb regle) · per calibrar i entrenar

Baies soltes sobre fons blanc amb un **regle**. Detecta **totes les baies** (sense delimitar racim). L'escala real en mm la posaràs després a l'anotador traçant sobre el regle.

In [ ]:
# A1) Carregar les fotos de safata des de Google Drive
from google.colab import drive
drive.mount('/content/drive')

import glob, os
CARPETA = '/content/drive/MyDrive/Mydrive racimos'   # ← posa aquí la teva carpeta
exts = ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG')
image_paths = sorted(p for e in exts for p in glob.glob(os.path.join(CARPETA, e)))
print(len(image_paths), 'imatges trobades'); print(image_paths[:5])

In [ ]:
# A2) Detectar baies a la SAFATA per visio classica (OpenCV: verd + watershed). Sense skimage.
import cv2, numpy as np

CALIBRA_1     = True       # True = nomes la imatge PREVIEW_IDX | False = totes
PREVIEW_IDX   = 0
HUE_MIN, HUE_MAX = 30, 95   # verd OpenCV (0-179). Baixa HUE_MIN si les baies son groguenques
SAT_MIN, VAL_MIN = 35, 35   # descarta blanc/gris del paper
MIN_AREA_FRAC = 0.00004     # area minima de baia
MAX_AREA_FRAC = 0.05        # area maxima (descarta tija/grumolls)
ASPECT_MAX    = 2.2         # descarta formes allargades (tija/branques/regle)
SEED_FRAC     = 0.5         # llavor del watershed (mes baix separa mes)
PEAK_K        = 1.2         # separacio minima entre baies (x radi tipic)

def _empty(path, img, W, H):
    return {'nombre': path, 'dataURL': data_url(img), 'w': W, 'h': H, 'escala_mm_px': None,
            'escalaLinea': None, 'bayas': [], 'racimo': None,
            'ficha': dict(EMPTY_FICHA), 'verdad': dict(EMPTY_VERDAD)}

def process_tray(path):
    img = open_img(path); W, H = img.size
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (HUE_MIN, SAT_MIN, VAL_MIN), (HUE_MAX, 255, 255))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)

    amax = MAX_AREA_FRAC * W * H
    nC, lab, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8))
    clean = np.zeros_like(mask)
    for i in range(1, nC):
        area = stats[i, cv2.CC_STAT_AREA]
        w_, h_ = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        if area > amax and max(w_, h_) / max(1, min(w_, h_)) > 2.5:
            continue
        clean[lab == i] = 255
    mask = clean

    dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    if dist.max() < 2:
        return _empty(path, img, W, H)
    hi = dist[dist > 0.5 * dist.max()]
    r0 = max(4.0, float(np.median(hi)) if hi.size else float(dist.max()))   # radi tipic (auto)

    ks = int(max(3, round(r0 * PEAK_K))); ks += (ks + 1) % 2                 # senar
    dil = cv2.dilate(dist, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ks, ks)))
    peaks = ((dist >= dil - 1e-3) & (dist > SEED_FRAC * r0)).astype(np.uint8)
    npk, peak_lbl = cv2.connectedComponents(peaks)
    if npk <= 1:
        return _empty(path, img, W, H)

    markers = np.zeros(dist.shape, np.int32)
    markers[cv2.erode((mask == 0).astype(np.uint8), k, iterations=2) > 0] = 1   # fons
    for lid in range(1, npk):
        markers[peak_lbl == lid] = lid + 1
    markers = cv2.watershed(bgr, markers)

    amin = MIN_AREA_FRAC * W * H
    bayas = []
    for lid in range(2, int(markers.max()) + 1):
        ys, xs = np.where(markers == lid)
        a = len(xs)
        if a < amin or a > amax:
            continue
        wbb = xs.max() - xs.min() + 1; hbb = ys.max() - ys.min() + 1
        if max(wbb, hbb) / max(1, min(wbb, hbb)) > ASPECT_MAX:
            continue
        r = float(np.sqrt(a / np.pi))
        bayas.append({'cx': round(float(xs.mean()), 2), 'cy': round(float(ys.mean()), 2), 'r': round(r, 2)})

    return {'nombre': path, 'dataURL': data_url(img), 'w': W, 'h': H, 'escala_mm_px': None,
            'escalaLinea': None, 'bayas': bayas, 'racimo': None,
            'ficha': dict(EMPTY_FICHA), 'verdad': dict(EMPTY_VERDAD)}

paths = [image_paths[PREVIEW_IDX]] if CALIBRA_1 else image_paths
images_out, fallades = [], []
for n, path in enumerate(paths, 1):
    try:
        im = process_tray(path); images_out.append(im)
        print(f"[{n}/{len(paths)}] {os.path.basename(path)}: {len(im['bayas'])} baies")
    except Exception as e:
        fallades.append(path); print(f"[{n}/{len(paths)}] {path}: ERROR ({e})")
if fallades: print('Fallades:', fallades)
save_project(images_out)


### A2b) Processar per lots de 20 (opcional, per a moltes fotos)
Deixa totes les fotos a la mateixa carpeta i canvia `BATCH` (1, 2, 3...) per fer-les de 20 en 20. Cada lot genera i descarrega el seu `modelvinya_loteN.json`.

In [ ]:
# A2b) Processar per LOTS (sense moure fitxers). Executa A1 i A2 abans (defineixen image_paths i process_tray).
BATCH_SIZE = 20
BATCH      = 1          # 1 = imatges 1-20, 2 = 21-40, 3 = 41-60, ...

start = (BATCH - 1) * BATCH_SIZE
chunk = image_paths[start:start + BATCH_SIZE]
print(f"Lot {BATCH}: {len(chunk)} imatges ({start+1}-{start+len(chunk)} de {len(image_paths)})")

images_out = []
for n, path in enumerate(chunk, 1):
    try:
        im = process_tray(path); images_out.append(im)
        print(f"  [{n}/{len(chunk)}] {os.path.basename(path)}: {len(im['bayas'])} baies")
    except Exception as e:
        print(f"  [{n}/{len(chunk)}] {path}: ERROR ({e})")

save_project(images_out, name=f'modelvinya_lote{BATCH}.json')

from google.colab import files
files.download(f'modelvinya_lote{BATCH}.json')


In [ ]:
# A3) Previsualitzar la primera imatge (comprova abans de baixar)
preview(images_out[0])

In [ ]:
# A4) Descarregar el projecte (.json)
from google.colab import files
files.download('modelvinya_sam3_proyecto.json')

---
## 🟩 OPCIÓ B — Racims a la planta (camp)

Racims penjats. SAM 3 delimita el **racim principal** (la màscara més gran) i només es queden les baies de dins. Sense referència real: les mides surten en px (l'escala la calibres amb les fotos de safata).

In [ ]:
# B1) Carregar les fotos de camp des de Google Drive
import glob, os
CARPETA_CAMP = '/content/drive/MyDrive/camp'   # ← carpeta de fotos de camp
exts = ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG')
image_paths = sorted(p for e in exts for p in glob.glob(os.path.join(CARPETA_CAMP, e)))
print(len(image_paths), 'imatges trobades'); print(image_paths[:5])

In [ ]:
# B2) Detectar baies del racim principal (ROI) + maxima cobertura
from scipy.ndimage import binary_dilation, label

CALIBRA_1B     = True        # True = nomes la 1a foto | False = totes
CLUSTER_PROMPT = 'bunch of grapes'
BERRY_PROMPT   = 'grape'
ROI_MARGIN_PX  = 15
SCORE_MIN_IN   = 0.30
AREA_MIN_FRAC2 = 0.000008
AREA_MAX_FRAC2 = 0.05
TILES          = 2           # franges verticals (3 si en falten a baix)
OVERLAP        = 0.15
DEDUP_FRAC     = 0.6
MIN_R_RATIO    = 0.55

processor.set_confidence_threshold(0.05)

def cluster_roi(img):
    masks, _ = detect(img, CLUSTER_PROMPT)
    if len(masks) == 0: return None
    bins = [to_binary(m) for m in masks]
    roi  = bins[int(np.argmax([b.sum() for b in bins]))]
    lbl, n = label(roi)
    if n > 1:
        sizes = np.bincount(lbl.ravel()); sizes[0] = 0; roi = (lbl == int(sizes.argmax()))
    return roi

def roi_to_polygon(roi):
    try:
        import cv2
        cnts, _ = cv2.findContours(roi.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: return None
        big = max(cnts, key=cv2.contourArea)
        approx = cv2.approxPolyDP(big, 0.005 * cv2.arcLength(big, True), True)
        pts = [[float(p[0][0]), float(p[0][1])] for p in approx]
        return {'puntos': pts, 'closed': True} if len(pts) >= 3 else None
    except Exception:
        return None

def process_field(path):
    img = open_img(path); W, H = img.size
    roi = cluster_roi(img)
    roi_d = binary_dilation(roi, iterations=ROI_MARGIN_PX) if (roi is not None and ROI_MARGIN_PX > 0) else roi
    if roi is not None:
        ys, xs = np.nonzero(roi)
        x0, x1 = max(0, int(xs.min())-20), min(W, int(xs.max())+20)
        y0, y1 = max(0, int(ys.min())-20), min(H, int(ys.max())+20)
    else:
        x0, y0, x1, y1 = 0, 0, W, H
    amin, amax = AREA_MIN_FRAC2 * W * H, AREA_MAX_FRAC2 * W * H
    th = (y1 - y0) / TILES
    cands = []
    for k in range(TILES):
        ty0 = max(y0, int(y0 + k*th - OVERLAP*th)); ty1 = min(y1, int(y0 + (k+1)*th + OVERLAP*th))
        masks, scores = detect(img.crop((x0, ty0, x1, ty1)), BERRY_PROMPT)
        for i in range(len(masks)):
            sc = float(scores[i]) if i < len(scores) else 0.0
            if sc < SCORE_MIN_IN: continue
            c = mask_to_circle(to_binary(masks[i]))
            if c is None or c['area_px'] < amin or c['area_px'] > amax: continue
            cx, cy, r = c['cx']+x0, c['cy']+ty0, c['r']
            if roi_d is not None:
                iy, ix = int(round(cy)), int(round(cx))
                if not (0 <= iy < roi_d.shape[0] and 0 <= ix < roi_d.shape[1] and roi_d[iy, ix]): continue
            cands.append((cx, cy, r, sc))
    cands.sort(key=lambda t: -t[2])
    bayas = []
    for cx, cy, r, sc in cands:
        if not any((cx-b['cx'])**2 + (cy-b['cy'])**2 < (DEDUP_FRAC*b['r'])**2 for b in bayas):
            bayas.append({'cx': round(cx,2), 'cy': round(cy,2), 'r': round(r,2)})
    if bayas:
        rmed = float(np.median([b['r'] for b in bayas]))
        bayas = [b for b in bayas if b['r'] >= MIN_R_RATIO * rmed]
    return {'nombre': path, 'dataURL': data_url(img), 'w': W, 'h': H,
            'escala_mm_px': None, 'escalaLinea': None, 'bayas': bayas,
            'racimo': roi_to_polygon(roi_d) if roi_d is not None else None,
            'ficha': dict(EMPTY_FICHA), 'verdad': dict(EMPTY_VERDAD)}

paths = image_paths[:1] if CALIBRA_1B else image_paths
images_out, fallades = [], []
for n, path in enumerate(paths, 1):
    try:
        im = process_field(path); images_out.append(im)
        print(f"[{n}/{len(paths)}] {os.path.basename(path)}: {len(im['bayas'])} baies")
    except Exception as e:
        fallades.append(path); print(f"[{n}/{len(paths)}] {path}: ERROR ({e})")
if fallades: print('Fallades:', fallades)
save_project(images_out)

In [ ]:
# B3) Previsualitzar la primera imatge (racim + baies)
preview(images_out[0])

In [ ]:
# B4) Descarregar el projecte (.json)
from google.colab import files
files.download('modelvinya_sam3_proyecto.json')

---
## 🟧 OPCIÓ C — Màxima cobertura (molts grans exposats)

Per quan el racim és **molt dens** i les Opcions A/B es deixen grans. Fa servir SAM 3 amb una **graella fina 3×3** sobre el racim, **diversos prompts alhora** i **dedup**, per recuperar els grans que falten. Executa abans el PAS 4 i carrega les imatges (A1 o B1). Ajusta amb la preview (cel·la 3 / B3).

In [ ]:
# C) MAXIMA COBERTURA per a racims MOLT densos (SAM 3): ROI + graella 3x3 + multi-prompt + dedup
#    Requereix PAS 4 (helpers) i tenir image_paths carregat (A1 safata o B1 camp).
from scipy.ndimage import binary_dilation, label

CALIBRA_1C       = True
PREVIEW_IDXC     = 0
CLUSTER_PROMPT_C = 'bunch of grapes'
BERRY_PROMPTS    = ['grape', 'grapes', 'green berry']   # uneix resultats de diversos prompts
ROI_MARGIN_PXC   = 15
SCORE_MIN_C      = 0.25       # baix = maxima cobertura
AREA_MIN_FRAC_C  = 0.000006
AREA_MAX_FRAC_C  = 0.05
GX, GY           = 3, 3       # graella fina sobre el racim (prova 4,4 si encara en falten)
OVERLAP_C        = 0.15
DEDUP_FRAC_C     = 0.6
MIN_R_RATIO_C    = 0.45

processor.set_confidence_threshold(0.05)

def cluster_roi_c(img):
    masks, _ = detect(img, CLUSTER_PROMPT_C)
    if len(masks) == 0: return None
    bins = [to_binary(m) for m in masks]
    roi = bins[int(np.argmax([b.sum() for b in bins]))]
    lbl, n = label(roi)
    if n > 1:
        sizes = np.bincount(lbl.ravel()); sizes[0] = 0; roi = (lbl == int(sizes.argmax()))
    return roi

def roi_poly_c(roi):
    try:
        import cv2
        cnts, _ = cv2.findContours(roi.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: return None
        big = max(cnts, key=cv2.contourArea)
        ap = cv2.approxPolyDP(big, 0.005 * cv2.arcLength(big, True), True)
        pts = [[float(p[0][0]), float(p[0][1])] for p in ap]
        return {'puntos': pts, 'closed': True} if len(pts) >= 3 else None
    except Exception:
        return None

def process_dense(path):
    img = open_img(path); W, H = img.size
    roi = cluster_roi_c(img)
    roi_d = binary_dilation(roi, iterations=ROI_MARGIN_PXC) if (roi is not None and ROI_MARGIN_PXC > 0) else roi
    if roi is not None:
        ys, xs = np.nonzero(roi)
        x0, x1 = max(0, int(xs.min()) - 20), min(W, int(xs.max()) + 20)
        y0, y1 = max(0, int(ys.min()) - 20), min(H, int(ys.max()) + 20)
    else:
        x0, y0, x1, y1 = 0, 0, W, H
    amin, amax = AREA_MIN_FRAC_C * W * H, AREA_MAX_FRAC_C * W * H
    tw, thh = (x1 - x0) / GX, (y1 - y0) / GY
    cands = []
    for gy in range(GY):
        for gx in range(GX):
            cx0 = max(x0, int(x0 + gx*tw - OVERLAP_C*tw)); cx1 = min(x1, int(x0 + (gx+1)*tw + OVERLAP_C*tw))
            cy0 = max(y0, int(y0 + gy*thh - OVERLAP_C*thh)); cy1 = min(y1, int(y0 + (gy+1)*thh + OVERLAP_C*thh))
            crop = img.crop((cx0, cy0, cx1, cy1))
            for pr in BERRY_PROMPTS:
                masks, scores = detect(crop, pr)
                for i in range(len(masks)):
                    sc = float(scores[i]) if i < len(scores) else 0.0
                    if sc < SCORE_MIN_C: continue
                    c = mask_to_circle(to_binary(masks[i]))
                    if c is None or c['area_px'] < amin or c['area_px'] > amax: continue
                    X, Y = c['cx'] + cx0, c['cy'] + cy0
                    if roi_d is not None:
                        iy, ix = int(round(Y)), int(round(X))
                        if not (0 <= iy < roi_d.shape[0] and 0 <= ix < roi_d.shape[1] and roi_d[iy, ix]): continue
                    cands.append((X, Y, c['r'], sc))
    cands.sort(key=lambda t: -t[2])
    bayas = []
    for X, Y, r, sc in cands:
        if not any((X - b['cx'])**2 + (Y - b['cy'])**2 < (DEDUP_FRAC_C * b['r'])**2 for b in bayas):
            bayas.append({'cx': round(X, 2), 'cy': round(Y, 2), 'r': round(r, 2)})
    if bayas:
        rmed = float(np.median([b['r'] for b in bayas]))
        bayas = [b for b in bayas if b['r'] >= MIN_R_RATIO_C * rmed]
    return {'nombre': path, 'dataURL': data_url(img), 'w': W, 'h': H, 'escala_mm_px': None,
            'escalaLinea': None, 'bayas': bayas, 'racimo': roi_poly_c(roi_d) if roi_d is not None else None,
            'ficha': dict(EMPTY_FICHA), 'verdad': dict(EMPTY_VERDAD)}

paths = [image_paths[PREVIEW_IDXC]] if CALIBRA_1C else image_paths
images_out, fallades = [], []
for n, path in enumerate(paths, 1):
    try:
        im = process_dense(path); images_out.append(im)
        print(f"[{n}/{len(paths)}] {os.path.basename(path)}: {len(im['bayas'])} baies")
    except Exception as e:
        fallades.append(path); print(f"[{n}/{len(paths)}] {path}: ERROR ({e})")
if fallades: print('Fallades:', fallades)
save_project(images_out)


---
## ✅ Després, a l'anotador
1. Obre **https://planessoria-ui.github.io/ModelVinya/** → **📦 Abrir proyecto** → tria el `.json`.
2. Revisa els cercles de cada imatge (afegeix/esborra/ajusta).
3. **Safata:** posa l'**escala (📏)** sobre el regle → diàmetres en mm. **Camp:** dibuixa/ajusta el racim i usa **✂ Fuera del racimo** si cal.
4. Omple la **ficha** i la **verdad terreno** (diàmetre real, etc.).
5. Exporta **🗂 YOLO dataset (.zip)** i **📄 CSV** per entrenar.

### Calibrar primer (recomanat)
A A2/B2 deixa `CALIBRA_1 = True` per provar amb **una sola foto** i ajustar prompt/llindars amb la previsualització. Quan vagi bé, posa `CALIBRA_1 = False` i torna a executar per processar **totes**.